In [ ]:
import pandas as pd
import geopandas as gpd
import glob

In [ ]:
#국토부 주민등록인구 100M 격자 데이터
gongju_grid100 = gpd.read_file('../P1_Data/01.External/국토부/국토부_공주시_100m격자_총인구수/nlsp_021001001.shp', encoding='utf-8')
gongju_grid100

In [ ]:
## 전국 센서스 행정경계 데이터
sensus = gpd.read_file('../P1_Data/01.External/국토부/Z_SOP_BND_ADM_DONG_PG/Z_SOP_BND_ADM_DONG_PG.shp',encoding='cp949')
gongju = sensus[sensus['ADM_CD'].astype(str).str.startswith('3402')]
gongju.to_file('../P1_Data/02.Pretreatment/gongju_boundary.shp', encoding = 'cp949')
gongju

In [ ]:
#좌표계 통일
gongju = gongju.to_crs(gongju_grid100.crs)

mapping = gpd.sjoin(gongju_grid100, gongju,how='left')

# 특정 컬럼 값이 중복되는 행 추출
duplicate_rows = mapping[mapping.duplicated(subset='gid', keep=False)]
#mapping한 기존격자에서 중복 되는 행 제거
mapping = mapping.drop_duplicates(subset='gid', keep=False)

##중복되는 행의 처리를 위한 작업
#중복되는 행들과 시흥 경계면 오버레이하여 걸쳐진 부분 면적 구함
res_union = gpd.overlay(duplicate_rows, gongju, how='intersection')

# 두 컬럼의 값이 동일한 행 추출
same_res_union = res_union[res_union['ADM_NM_1'] == res_union['ADM_NM_2']]

#면적계산
same_res_union['area'] = same_res_union['geometry'].area

# 같은 GID 값끼리 그룹화하여 면적컬럼에서 가장 큰 값을 가진 읍면동컬럼으로 업데이트
result_union = same_res_union.groupby('gid').apply(lambda x: x.loc[x['area'].idxmax()]).reset_index(drop=True)

# 행정동 매핑한 격자와 여러개의 행정동이 걸처진 격자처리 후 두개의 데이터프레임을 다시 합치는 과정
mapping = mapping[['gid','ADM_CD','ADM_NM','val','geometry']]

result_union = result_union[['gid', 'ADM_CD_1', 'ADM_NM_1', 'val', 'geometry']]
# 'A' 컬럼의 이름을 'X'로 변경
result_union.rename(columns={'ADM_CD_1': 'ADM_CD'}, inplace=True)
result_union.rename(columns={'ADM_NM_1': 'ADM_NM'}, inplace=True)


#결과
gongju_grid_regispop = pd.concat([mapping, result_union], axis=0)
# 인덱스를 리셋합니다.
gongju_grid_regispop = gongju_grid_regispop.reset_index(drop=True)
gongju_grid_regispop

In [ ]:
gongju_grid_regispop.to_file('../P3_Result/03.SHP/공주시_100M격자_주민등록인구(행정동)_2023년도.shp', encoding = 'cp949')

In [ ]:
path_data = 'H:/23project/A11/P1_Data/01.External/공주시_생활인구데이터/'

In [ ]:
# SKT 폴더내 SEX_AGE_PCELL_POP 데이터 병합
flow_list = glob.glob(path_data + 'SKT/*/*SEX_AGE_PCELL_POP*.csv')
flow_df = pd.DataFrame()
for flow_path in flow_list:
    temp = pd.read_csv(flow_path, dtype={"COORD_X" : 'str', "COORD_Y" : 'str'})
    flow_df = pd.concat([flow_df,temp],axis=0)

In [ ]:
flow_df

### 국토부

In [ ]:
# 전국 센서스 행정경계 데이터
sensus = gpd.read_file('../P1_Data/01.External/국토부/Z_SOP_BND_ADM_DONG_PG/Z_SOP_BND_ADM_DONG_PG.shp',encoding='cp949')
sensus

In [ ]:
# 공주시 추출
gongju = sensus[sensus['ADM_CD'].str.startswith('34020')]
#gongju.to_file('../P1_Data/02.Pretreatment/gongju_boundary.shp', encoding = 'cp949')
gongju

In [ ]:
# 전국 센서스 100M 격자 데이터
GRID = gpd.read_file('../P1_Data/01.External/국토부/국토부_공주시_100m격자_총인구수/nlsp_021001001.shp',encoding='utf-8')
GRID